In [8]:
from google.colab import files
uploaded = files.upload()

Saving dataset_20k_youtube.csv to dataset_20k_youtube.csv


In [11]:
import pandas as pd

df = pd.read_csv('dataset_20k_youtube.csv')

In [26]:
df.head()

,comment_id,parent_id,author,comment,likes,published,is_reply,label
0,UgxI4XpwhOHq1ugcCF54AaABAg,NaN,@tikanursantika,yuka mending kamu nikah aja deh ma jule biar t...,0,2026-03-25T07:27:02Z,False,neutral
1,UgyYC1zja9DTIu0t-1Z4AaABAg,NaN,@Keysa-w1z,jule go internasional anjr,0,2026-03-25T06:06:05Z,False,neutral
2,UgwUG1-6e11A_XekNQh4AaABAg,NaN,@user-aj16,ngomong kaya org takut salah ngomongduduk kada...,0,2026-03-22T23:31:32Z,False,neutral
3,UgxHLWgkpo8ljtjEAs94AaABAg,NaN,@nofiaFia-h2c,sama ini nih,0,2026-03-22T05:48:59Z,False,neutral
4,UgyDzaolw8JsUCW8Fql4AaABAg,NaN,@nofiaFia-h2c,aku punya lagu buat jule,0,2026-03-22T05:48:19Z,False,neutral


In [14]:
df.columns

Index(['comment_id', 'parent_id', 'author', 'comment', 'likes', 'published',
       'is_reply'],
      dtype='object')

In [15]:
#fungsi untuk membersihkan teks
import re
import string

def cleaning_text(text):
    text = str(text)

    text = re.sub(r'@\w+', '', text)  # hapus mention
    text = re.sub(r'#\w+', '', text)  # hapus hashtag
    text = re.sub(r'http\S+|www\S+', '', text)  # hapus URL
    text = re.sub(r'RT[\s]+', '', text)  # hapus RT
    text = re.sub(r'[0-9]+', '', text)  # hapus angka
    text = re.sub(r'[^\w\s]', '', text)  # hapus simbol
    text = text.replace('\n', ' ')  # newline jadi spasi
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()

    return text

In [16]:
slang_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "lu": "kamu",
    "gw": "saya",
    "anj": "anjing",
    "gblk": "goblok"
}

def normalize_slang(text):
    words = text.split()
    normalized = [slang_dict.get(word, word) for word in words]
    return " ".join(normalized)

In [17]:
def case_folding(text):
    return text.lower().strip()

In [18]:
def preprocess(text):
    text = cleaning_text(text)
    text = normalize_slang(text)
    text = case_folding(text)
    return text

In [20]:
df['comment'] = df['comment'].apply(preprocess)

In [21]:
df[['comment']].head()

,comment
0,yuka mending kamu nikah aja deh ma jule biar t...
1,jule go internasional anjr
2,ngomong kaya org takut salah ngomongduduk kada...
3,sama ini nih
4,aku punya lagu buat jule


In [22]:
hate_words = [
    'anjing','bangsat','tolol','goblok','idiot','bodoh'
]

threat_words = [
    'bunuh','matiin','hajar','bacok','gebuk'
]

sexual_words = [
    'ngentot','sange','mesum','bokep'
]

positive_words = [
    'bagus','keren','mantap','hebat','love','suka'
]

In [23]:
def label_intent(text):
    text = str(text).lower()

    if any(word in text for word in threat_words):
        return "threat"
    elif any(word in text for word in sexual_words):
        return "sexual_harassment"
    elif any(word in text for word in hate_words):
        return "insult"
    elif any(word in text for word in positive_words):
        return "positive"
    else:
        return "neutral"

In [27]:
df['label'] = df['comment'].apply(label_intent)

In [28]:
df['label'].value_counts()

,count
label,
neutral,18870
insult,850
positive,258
threat,21
sexual_harassment,1


In [29]:
df.to_csv("dataset_labeled.csv", index=False)

In [30]:
from google.colab import files
files.download("dataset_labeled.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>